# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [8]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:8888",  # or your app/site URL
        "X-Title": "llm_engineering_course"       # optional label
    }
)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [9]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [10]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [11]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [12]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [13]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [16]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/news page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [17]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [18]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [19]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Zhihu', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'Discuss community', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'API Endpoints', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [20]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [21]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-397B-A17B
Updated
3 days ago
•
303k
•
907
nvidia/personaplex-7b-v1
Updated
8 days ago
•
539k
•
2.15k
Nanbeige/Nanbeige4.1-3B
Updated
1 day ago
•
178k
•
725
zai-org/GLM-5
Updated
10 days ago
•
179k
•
1.45k
MiniMaxAI/MiniMax-M2.5
Updated
7 days ago
•
209k
•
863
Browse 2M+ models
Spaces
Running
on
Zero
MCP
874
Wan2.2 14B Preview
🐌
874
generate a video from an image with a text prompt
Running
on
Zero
MCP
2.

In [31]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGGML and llama.cpp join Hugging Face 🔥\nTry HuggingChat Omni – Chat with AI 💬\nGet started with Inference in seconds 🚀\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-397B-A17B\nUpdated\n3 days ago\n•\n303k\n•\n907\nnvidia/personaplex-7b-v1\nUpdated\n8 days ago\n•\n539k\n•\n2.15k\nNanbeige/Nanbeige4.1-3B\nUpdated\n1 day ago\n•\n178k\n•\n726\nzai-org/GLM-5\nUpdated\n10 days ago\n•\n179k\n•\n1.45k\nMiniMaxAI/MiniMax-M2.5

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [28]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a leading AI community and collaboration platform dedicated to building the future of machine learning (ML). It serves as a vibrant hub where the global ML community comes together to share, explore, and experiment with machine learning models, datasets, and applications.

At its core, Hugging Face empowers machine learning engineers, scientists, and end users alike to collaborate openly and ethically on cutting-edge AI technologies. It fosters innovation by offering access to over 2 million models, 500,000+ datasets, and a rich ecosystem of AI-powered applications.

---

## What Hugging Face Offers

- **Collaboration Platform:** Host and collaborate on unlimited public models, datasets, and applications.
- **Models:** Access and contribute to a vast catalog of ML models spanning text, image, video, audio, and even 3D modalities.
- **Datasets:** Explore hundreds of thousands of datasets curated for various AI research and applications.
- **Spaces:** Showcase interactive AI apps and demos created by the community.
- **Tools and Libraries:** Utilize their open-source stack to accelerate ML development and deployment.
- **Enterprise Solutions:** Accelerate your AI transformation with paid compute options and enterprise-focused services.

---

## Community and Culture

Hugging Face prides itself on having a fast-growing, dynamic community characterized by openness, inclusivity, and ethical AI development. The platform acts as a nurturing environment for the next generation of ML talent to learn, build portfolios, and share breakthroughs.

The culture is built on collaboration — encouraging users to contribute models, datasets, and applications, or engage with others’ work to push forward AI advances together.

---

## Who Uses Hugging Face?

- **Machine Learning Engineers & Researchers:** For sharing, experimenting, and deploying ML models across various domains.
- **AI-Driven Businesses:** To accelerate AI initiatives using robust enterprise tools and compute resources.
- **Developers & Data Scientists:** To access ready-to-use models and datasets for rapid prototyping and production.
- **Students & Educators:** To learn from the community and build practical ML experience through hands-on collaboration.
- **AI Enthusiasts:** To explore and interact with cutting-edge AI apps and demos hosted on the platform.

---

## Careers at Hugging Face

Hugging Face offers exciting career opportunities for passionate individuals who want to contribute to the future of AI. They seek talents who embrace collaboration, open-source values, and innovation in AI.

As a community-driven company, Hugging Face supports professional growth by allowing employees to work at the forefront of machine learning research, development, and application.

---

## Get Started with Hugging Face

Join a global community dedicated to the open and ethical advancement of machine learning.

- Explore over **2 million models**
- Browse **500,000+ datasets**
- Discover and create AI applications in the **Spaces** environment
- Accelerate your projects using the **open-source ML stack**
- Build your profile and showcase your work

Sign up today at [huggingface.co](https://huggingface.co) to start collaborating and innovate with the future of AI.

---

## Brand and Identity

- **Logo Colors:** Yellow (#FFD21E), Orange (#FF9D00), Gray (#6B7280)
- Hugging Face’s branding embodies openness, energy, and community spirit aligning with its mission to democratize AI.

---

Hugging Face is not just a platform but a movement—building an open, powerful, and ethical AI ecosystem for everyone. Join the revolution in machine learning today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [29]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a vibrant, open, and innovative AI community dedicated to shaping the future of machine learning. It serves as a collaborative platform where researchers, developers, and enterprises come together to create, discover, and share state-of-the-art models, datasets, and applications across multiple AI modalities including text, image, video, audio, and even 3D.

---

## The Platform

- **Models:** Access over 2 million machine learning models, including trending and cutting-edge architectures like Qwen/Qwen3.5, PersonaPlex, and GLM-5.
- **Datasets:** Browse and contribute to a collection of more than 500,000 datasets covering diverse tasks and domains.
- **Spaces:** Host and interact with AI-powered applications and demos such as image generation from text, video creation from images, custom speech generation, and image editing tools—all powered by community-built models.
- **Compute and Enterprise Solutions:** Accelerate your ML projects with Hugging Face’s paid compute resources and enterprise offerings for scalable AI deployment.

---

## Company Culture & Community

Hugging Face thrives on open collaboration and community contribution. The active user base of over 83,000 followers participates daily—updating datasets, sharing feedback, and pushing the boundaries of AI research. The company values transparency, inclusivity, and innovation, providing an environment where both newcomers and experts can build their portfolios and gain recognition in the AI field.

---

## Customers & Users

- **Researchers and Academics:** Utilize Hugging Face’s vast repository of models and datasets to explore new AI frontiers.
- **Developers:** Build innovative AI applications with accessible tools and hosted Spaces.
- **Enterprises:** Leverage Hugging Face’s enterprise solutions to integrate and deploy AI at scale.
- **AI Enthusiasts and Hobbyists:** Engage with a supportive community, explore applications, and showcase their ML projects.

---

## Careers at Hugging Face

Join Hugging Face to work at the forefront of AI innovation. The company offers opportunities across engineering, research, product management, and community engagement roles. Employees are part of a fast-paced, collaborative environment focused on open-source principles and shaping the future of machine learning technology.

For more information on current job openings and how to apply, visit Hugging Face’s official careers page on their website.

---

## Get Started Today

- Explore over **2 million models**, **500,000+ datasets**, and **1 million+ AI applications**.
- Join a global AI community passionate about innovation and collaboration.
- Build, share, and accelerate your machine learning projects with industry-leading tools and support.

Sign up at [huggingface.co](https://huggingface.co) to start your journey with Hugging Face—the home of machine learning.

---

*Hugging Face: Empowering the AI community to build the future.*

In [32]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face: Where AI Gets Cozy and Brilliant 🤗

Welcome to **Hugging Face**, the AI community that's not just building the future — we're hugging it into shape! If you think AI is all serious business and no fun, think again. Here at Hugging Face, we blend cutting-edge machine learning with collaboration, creativity, and a sprinkle of charm.

---

## What We Are: The Home of Machine Learning Magic

Imagine a giant playground for AI enthusiasts — from the newbie coder to the PhD data wizard. We provide a **collaboration platform** where you can:

- Explore over **2 million models** (yes, 2 million!).
- Dive into **500k+ datasets** covering everything from text to 3D objects.
- Build and share **1 million+ AI-powered applications**.
- Host your own projects and **build a shiny ML portfolio** to dazzle the world.

Whether you're into text, images, audio, video, or even quirky 3D creations, Hugging Face is your creative lab.

---

## Our Community: The AI Dream Team

With a **fast-growing global community** powered by some of the most popular open-source libraries and tech, we're not just a company — we're a movement. Scientists, engineers, artists, and dreamers join forces to create an **open, ethical AI future**.

We cheer for collaboration, transparency, and innovation — because making AI better requires all minds and hugs on deck.

---

## Enterprise Might: AI That Means Business

For companies wanting AI at scale but with a human touch, our **Enterprise Hub** offers:

- Rock-solid **security** with single sign-on (SSO), audit logs, and granular access controls.
- Flexible **compute** options with ZeroGPU boosts so your AI runs faster than your last cup of coffee.
- Handy **analytics** dashboards that keep you in the know (because guessing is so 2020).
- Generous **private storage** space — no more fretting over where to stash those terabytes of AI genius.

Starting at just $20/user/month for teams, or tailored contracts for enterprises, it’s AI peace of mind with a high-five.

---

## Career Opportunities: Join Our Hug Squad 🤗

Want to work where the future of AI meets a friendly, inclusive culture? Hugging Face is the place to grow your skills, collaborate with brilliant minds, and push the boundaries of technology — all while having plenty of fun.

We champion:

- Innovation-driven work environments.
- Open-source contributions that make real-world impacts.
- A culture that values your voice, creativity, and enthusiasm for AI.

Keep an eye on our **Careers page** — your next big adventure awaits.

---

## Why Hugging Face?  

Because who says cutting-edge AI research can’t come with a smile? From open-source pioneers to enterprise heroes, we unite people and tech to create a more connected, intelligent future. Plus, if you love adorable logos and warm fuzzy vibes with your tech, you’re home.

---

### Quick Hug Summary

| What?                   | Hugging Face Magic                      |
|-------------------------|---------------------------------------|
| Models                  | 2M+ open-source models to explore     |
| Datasets                | 500k+ datasets, ready for action      |
| Applications            | 1M+ AI apps built by creative minds   |
| Enterprise Plans        | Scalable, secure, and smart            |
| Community               | Global, fast-growing, open, ethical    |
| Careers                 | Join the AI hug squad, innovate freely|

Ready to explore? Come for the AI, stay for the hugs.

**Sign up now** and get cozy with the future at [huggingface.co](https://huggingface.co)

---

*Powered by passion, smarts, and hugs.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>